# Datenbereinigung

In [5]:
import pandas as pd

df = pd.read_csv('../data/RTA_Dataset.csv')

In [6]:
df_clean = df.copy()

# Spalten mit Text auswählen (außer Uhrzeit)
text_cols = df_clean.select_dtypes(exclude='number').columns.drop('Time')

# 1. Leerzeichen entfernen
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# 2. Fehlende Werte vereinheitlichen
df_clean[text_cols] = df_clean[text_cols].replace(
    ['unknown', 'na', 'Unknown or other'], 'Unknown')
df_clean[text_cols] = df_clean[text_cols].fillna('Unknown')

# 3. Fehlerhafte Werte korrigieren
df_clean['Type_of_vehicle'] = df_clean['Type_of_vehicle'].str.replace('?', '-', regex=False)
df_clean['Area_accident_occured'] = df_clean['Area_accident_occured'].replace(
    'Rural village areasOffice areas', 'Unknown')
df_clean['Lanes_or_Medians'] = df_clean['Lanes_or_Medians'].replace('other', 'Other')

# 4. Zielvariable: Serious + Fatal → schwer (1), Slight → 0
df_clean['schwer'] = (df_clean['Accident_severity'] != 'Slight Injury').astype(int)

In [7]:
# Kontrolle
for col in ['Area_accident_occured', 'Type_of_vehicle', 'Driving_experience']:
    print(col, sorted(df_clean[col].unique()))

Area_accident_occured ['Church areas', 'Hospital areas', 'Industrial areas', 'Market areas', 'Office areas', 'Other', 'Outside rural areas', 'Recreational areas', 'Residential areas', 'Rural village areas', 'School areas', 'Unknown']
Type_of_vehicle ['Automobile', 'Bajaj', 'Bicycle', 'Long lorry', 'Lorry (11-40Q)', 'Lorry (41-100Q)', 'Motorcycle', 'Other', 'Pick up upto 10Q', 'Public (12 seats)', 'Public (13-45 seats)', 'Public (> 45 seats)', 'Ridden horse', 'Special vehicle', 'Stationwagen', 'Taxi', 'Turbo', 'Unknown']
Driving_experience ['1-2yr', '2-5yr', '5-10yr', 'Above 10yr', 'Below 1yr', 'No Licence', 'Unknown']


### Durch die Datenbereinigung gelöste Probleme
1. Überflüssige Leerzeichen entfernt
2. Fehlende Werte vereinheitlicht: NaN, unknown, na und Unknown or other → Unknown
3. Kodierungsfehler korrigiert: Lorry (41?100Q) → Lorry (41-100Q)
4. Fehlerhafte Werte bereinigt: Rural village areasOffice areas → Unknown; other → Other
5. Zielvariable erstellt: schwer (1) = Serious + Fatal, leicht (0) = Slight

In [8]:
df_clean.to_parquet('../data/df_clean.parquet', index=False)
print('Gespeichert:', df_clean.shape)

Gespeichert: (12316, 33)
